<a href="https://colab.research.google.com/github/Prajkta11222/Iot_Based_Smart_Health_Monetoring_System/blob/main/ECG_ML_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#IoT Smart Health Monitoring System

In [ ]:
# =========================
# STAGE 1: LOAD DATA
# =========================

from google.colab import drive
drive.mount("/content/drive")

import os
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/iot_dataset"

TRAIN_PATH = f"{BASE}/mitbih_train.csv"
TEST_PATH  = f"{BASE}/mitbih_test.csv"

assert os.path.exists(TRAIN_PATH), f"Train file not found: {TRAIN_PATH}"
assert os.path.exists(TEST_PATH), f"Test file not found: {TEST_PATH}"

train = pd.read_csv(TRAIN_PATH, header=None)
test  = pd.read_csv(TEST_PATH, header=None)

print("Train shape:", train.shape)
print("Test shape :", test.shape)

LABEL = train.columns[-1]
ECG_COLS = train.columns[:-1]

print("\nOriginal classes:")
print(train[LABEL].value_counts().sort_index())

print("\nMissing values:", train.isna().sum().sum())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train shape: (87554, 188)
Test shape : (21892, 188)

Original classes:
187
0.0    72471
1.0     2223
2.0     5788
3.0      641
4.0     6431
Name: count, dtype: int64

Missing values: 0


In [ ]:
# =========================
# STAGE 2: PREPARE DATA
# =========================

# Class mapping:
# 0 = Normal              -> LOW
# 1 = Supraventricular   -> HIGH
# 2 = Ventricular        -> HIGH
# 3 = Fusion             -> HIGH
# 4 = Unknown             -> EXCLUDED

RISK_MAP = {
    0: 0,   # LOW
    1: 1,   # HIGH
    2: 1,   # HIGH
    3: 1    # HIGH
}

train = train[train[LABEL] != 4].copy()
test  = test[test[LABEL] != 4].copy()

X = train[ECG_COLS].astype(np.float32)
y = train[LABEL].map(RISK_MAP).astype(np.int8)

X_test = test[ECG_COLS].astype(np.float32)
y_test = test[LABEL].map(RISK_MAP).astype(np.int8)

print("Training data:", X.shape)
print("Test data    :", X_test.shape)

print("\nLOW/HIGH distribution:")
print(pd.Series(y).map({0:"LOW", 1:"HIGH"}).value_counts())

Training data: (81123, 187)
Test data    : (20284, 187)

LOW/HIGH distribution:
187
LOW     72471
HIGH     8652
Name: count, dtype: int64


In [ ]:
# =========================
# STAGE 3: FAST ML TRAINING
# =========================

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

RANDOM_STATE = 42

# Stratified validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

# Faster baseline
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=100,       # 400 → 100 for faster training
        max_depth=20,           # limits unnecessary complexity
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

print("\nTraining Random Forest...")
model.fit(X_train, y_train)
print("Training complete.")

# Validation
val_prob = model.predict_proba(X_val)[:, 1]
val_pred = (val_prob >= 0.5).astype(np.int8)

print("\nVALIDATION RESULTS")
print("==================")

print(classification_report(
    y_val,
    val_pred,
    target_names=["LOW", "HIGH"],
    digits=4
))

print("ROC-AUC:",
      round(roc_auc_score(y_val, val_prob), 4))

print("PR-AUC:",
      round(average_precision_score(y_val, val_prob), 4))

Training samples: 68954
Validation samples: 12169

Training Random Forest...
Training complete.

VALIDATION RESULTS
              precision    recall  f1-score   support

         LOW     0.9804    0.9975    0.9889     10871
        HIGH     0.9756    0.8328    0.8986      1298

    accuracy                         0.9799     12169
   macro avg     0.9780    0.9152    0.9437     12169
weighted avg     0.9799    0.9799    0.9792     12169

ROC-AUC: 0.991
PR-AUC: 0.9634


In [ ]:
# =========================
# STAGE 4: FINAL TEST + SAVE
# =========================

import joblib
import json

# IMPORTANT:
# Test set is used only once for final evaluation.
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(np.int8)

print("FINAL TEST RESULTS")
print("==================")

print(classification_report(
    y_test,
    test_pred,
    target_names=["LOW", "HIGH"],
    digits=4
))

print("ROC-AUC:",
      round(roc_auc_score(y_test, test_prob), 4))

print("PR-AUC:",
      round(average_precision_score(y_test, test_prob), 4))

# Save model
MODEL_DIR = f"{BASE}/trained_model"
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(
    model,
    f"{MODEL_DIR}/ecg_risk_model.pkl"
)

metadata = {
    "model": "RandomForestClassifier",
    "version": "1.0",
    "input_samples": 187,
    "sampling_rate_hz": 125,
    "target": "risk_level",
    "labels": {
        "0": "LOW",
        "1": "HIGH"
    },
    "original_mapping": {
        "0": "LOW",
        "1": "HIGH",
        "2": "HIGH",
        "3": "HIGH",
        "4": "EXCLUDED"
    },
    "random_state": 42,
    "note": "ECG abnormality/research-risk classification, not medical diagnosis."
}

with open(f"{MODEL_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\nModel saved to:")
print(f"{MODEL_DIR}/ecg_risk_model.pkl")

FINAL TEST RESULTS
              precision    recall  f1-score   support

         LOW     0.9793    0.9977    0.9884     18118
        HIGH     0.9775    0.8232    0.8937      2166

    accuracy                         0.9791     20284
   macro avg     0.9784    0.9105    0.9411     20284
weighted avg     0.9791    0.9791    0.9783     20284

ROC-AUC: 0.9905
PR-AUC: 0.9613

Model saved to:
/content/drive/MyDrive/iot_dataset/trained_model/ecg_risk_model.pkl


In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/iot_dataset/trained_model/ecg_risk_model.pkl")
files.download("/content/drive/MyDrive/iot_dataset/trained_model/metadata.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>